# ARI of single-linkage clustering

The separation condition $\Delta_{\mathrm{out}} > \Delta_{\mathrm{in}}$ (Assumption 3) is
*sufficient* for single linkage to recover the true partition. `assumptions.ipynb` measured how
often it holds, and the answer was: not that often past $K = 4$. This notebook asks the converse
question — **how much of the recovery survives where the condition fails?** Nothing here is
conditioned on an assumption being satisfied: the mixture of the first figure is a first draw with
no selection, and the grid of the second one is the same $(\alpha, K)$ grid as in
`assumptions.ipynb`, kept in full.

**What is measured.** The Adjusted Rand Index between the true partition $P^\star$ and the
single-linkage partition, and separately the exact-recovery event $\{\mathrm{ARI} = 1\}$, which is
the event the consistency theorem bounds. ARI is the informative summary in the regime this
notebook is about: it degrades gracefully, so it distinguishes "two clusters merged" from
"partition unrelated to the truth", where the exact-recovery indicator sees only a failure.

**Design.**
* $\Sigma = \{0,\dots,4\}$, first-order chains, transition rows drawn from a Dirichlet($\alpha$)
  prior. $N$ sequences of length $n$ with i.i.d. latent labels, balanced weights $w_k = 1/K$
  (single linkage needs no balance condition). Every sequence starts from its own initial law, the
  Dirac law at a state drawn uniformly on $\Sigma$, as in the other two notebooks.
* Costs fixed in advance, hence deterministic as the theory requires: $c_{\mathrm{sub}} \equiv 2$,
  $\delta \equiv 1$, so $M = 2$.
* **Estimator: single linkage cut at $K$ blocks**, $K$ known. The consistency theorem is stated for
  the cut at a fixed level $t \in (\Delta_{\mathrm{in}}, \Delta_{\mathrm{out}})$, but its proof
  shows that on the event $\mathcal E_{N,n}(\varepsilon)$ the two cuts coincide; cutting at $K$
  avoids an oracle threshold that no practitioner could pick. When a class comes out empty the cut
  is taken at the number of non-empty classes, which is what $P^\star$ counts.
* $\mathrm{ARI}_n$ is evaluated on **nested prefixes** of the same $N$ trajectories, so a curve
  $n \mapsto \mathrm{ARI}_n$ is a genuine sample path of one dataset growing in length, not a
  sequence of independent draws.

**A by-product that costs nothing.** Once the $N \times N$ dissimilarity matrix is computed, the
true labels give a plug-in estimate of $\Gamma$: $\hat\Gamma_{k\ell}$ is the average of
$\hat\gamma_n(i,j)$ over the pairs of the block $(k,\ell)$, whence $\hat\Delta_{\mathrm{in}}$,
$\hat\Delta_{\mathrm{out}}$ and the separation margin $\hat\eta = \hat\Delta_{\mathrm{out}} -
\hat\Delta_{\mathrm{in}}$. The ARI and the margin are therefore measured **on the same data**, and
can be compared cell by cell. Note that this estimator averages over
$|C_k|(|C_k|-1)/2$ or $|C_k||C_\ell|$ pairs, whereas `assumptions.ipynb` uses one pair per block
(two independent realisations per component): it is much less noisy, so its numbers are not
expected to match that notebook digit for digit.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from om_lib import (PAPER_STYLE, SEQUENTIAL_CMAP, adjusted_rand_index,
                    check_assumption_metric, cost_scheme, cut_at_k, om_matrices,
                    sample_markov_model, sample_mixture, separation_levels,
                    single_linkage_tree, spectral_gap, stationary_distribution_markov)

## Setup

One evaluation function, shared by the two experiments: it computes the dissimilarity matrices on
the nested prefixes, then for each horizon the single-linkage ARI and the plug-in separation
levels.

In [ ]:
D_STATES = 5           # alphabet size
SEED     = 20260803

S_COST, DELTA_COST = cost_scheme("constant", D_STATES, sub=2.0, indel=1.0)
assert all(v for k, v in check_assumption_metric(S_COST, DELTA_COST).items() if k != "M = max c_sub")
EXACT_TOL = 1e-12      # ARI > 1 - EXACT_TOL is the exact-recovery event {ARI = 1}


def evaluate_paths(X, labels, grid, K):
    """Single-linkage ARI and separation levels along nested prefixes of X.

    Returns arrays indexed by the horizon grid: the ARI against the true labels, and the
    plug-in Delta_in, Delta_out, Delta_out^max. The cut is taken at the number of non-empty
    classes, which is the number of blocks of P*.
    """
    N = X.shape[0]
    K_eff = int(np.unique(labels).size)
    Ds = om_matrices(X, grid, S_COST, DELTA_COST)
    out = {key: np.empty(len(grid)) for key in ("ari", "in", "out", "out_max")}
    for g in range(len(grid)):
        _, edges = single_linkage_tree(Ds[g])
        out["ari"][g] = adjusted_rand_index(cut_at_k(edges, N, K_eff), labels)
        levels = separation_levels(Ds[g], labels, K)
        for key in ("in", "out", "out_max"):
            out[key][g] = levels[key]
    return out

## One axis for both $n$ and $N$

The exact-recovery bound of the paper is $N^2\exp(-\varepsilon^2 n / (2C^\star))$, which rewrites as

$$\exp\Big(2\log N \Big(1 - \frac{\lambda}{\lambda^\star}\Big)\Big),
  \qquad \lambda := \frac{n}{\log N}, \qquad \lambda^\star := \frac{4C^\star}{\varepsilon^2}.$$

The two sample sizes enter **only through the ratio $\lambda = n/\log N$**: the remaining $\log N$
multiplies the exponent, so it sharpens the transition but does not move it. One figure therefore
carries the dependence on both: mean ARI against $n$ on the left, one curve per $N$, and the *same*
points against $n/\log N$ on the right. Whether they fall on a single curve is the question the
right panel answers.

Two things make the comparison clean:

* **The mixture is the same for every $(N, n)$.** The $K$ kernels are drawn once — first draw, no
  selection — and only the labels and trajectories are redrawn. Everything the bound depends on
  besides $n$ and $N$ ($\varepsilon$ through the separation gap $\eta$, $C^\star$ through the mixing
  times) is thus held fixed, and $\lambda$ is the only free combination left.
* **The grid is in $\lambda$, not in $n$.** For each $N$ the horizons are $n = \lfloor\lambda\log
  N\rceil$ over a common $\lambda$ grid, so all the $N$ land on the same abscissae in the right panel
  by construction, and the compute budget adapts on its own — short sequences for small $N$, where
  the transition happens earlier.

The horizons of a given $N$ are again nested prefixes of the same trajectories, so a replicate is a
sample path. The band is the 10th-to-90th percentile over replicates; the number of replicates
decreases with $N$, since the ARI of a large sample is already an average over many pairs and varies
less.

**Why $K = 2$ here.** Measuring a threshold requires the threshold to sit inside the observation
window. At $K = 3$ an unselected Dirichlet draw is a lottery: over 9 draws at
$\alpha \in \{0.2, 0.3, 0.4\}$, the margin $\hat\eta$ ranged from $-0.001$ to $+0.397$ and three of
them never reached $\mathrm{ARI} = 0.99$ by $n = 800$, so $n^\star$ would fall outside any affordable
grid (the cost is $O(N^2n^2)$). All 9 draws at $K = 2$ reached it, between $n = 32$ and $n = 504$.
$K = 2$ is also the minimal instance of the theorem, and the scaling this figure is about comes from
the union bound over the $\binom{N}{2}$ pairs, which does not involve $K$ — the dependence on $K$ is
the subject of the grid below. The draw used is again the first one, unselected, and its margin is
printed.

In [ ]:
ALPHA_FIX = 0.3      # moderate randomness: distinguishable chains that still mix fast
K_FIX     = 2

#: lambda = n / log N, the rescaled horizon; common to every N. The upper end is set so that
#: the mean ARI has plateaued at 1: for this mixture ARI = 0.99 is reached around lambda = 30.
LAMBDAS = np.geomspace(3.0, 70.0, 14)
#: N ranges over a factor 40, i.e. a factor 2.6 in log N -- the lever of the rescaling.
#: Adding 1000 widens it to 3.0 but costs ~12 min on its own (N^2 pairs at R = 8).
N_LIST = [10, 25, 60, 150, 400]


def replicates_for(N):
    """Replicates at this N. Cost per replicate grows as N^2, and the ARI of a large sample
    concentrates, so trading replicates for N keeps both the cost and the precision even."""
    return max(8, int(round(1600 / N)))


rng = np.random.default_rng(SEED)
KERNELS = np.stack([sample_markov_model(D_STATES, 1, ALPHA_FIX, rng)["transitions"]
                    for _ in range(K_FIX)])

print("spectral gaps   :", np.round([spectral_gap(P) for P in KERNELS], 3))
print("stationary laws :\n", np.round(np.stack([stationary_distribution_markov(P)
                                                for P in KERNELS]), 3))
print("lambda grid     :", np.round(LAMBDAS, 1))

# where this mixture sits with respect to the separation condition, at a reference horizon
ref_mix = sample_mixture(K_FIX, 60, 800, D_STATES, ALPHA_FIX, rng, kernels=KERNELS)
ref = evaluate_paths(ref_mix["X"], ref_mix["labels"], [800], K_FIX)
print(f"at n = 800, N = 60: Delta_in = {ref['in'][0]:.3f}, Delta_out = {ref['out'][0]:.3f}, "
      f"margin = {ref['out'][0] - ref['in'][0]:+.3f}, ARI = {ref['ari'][0]:.3f}")

In [ ]:
def sweep_n_and_N(N_list, lambdas, kernels, alpha, K, seed=SEED + 1):
    """Mean ARI over a grid of (N, lambda), with n = round(lambda * log N).

    One entry per N, holding the horizons actually used, the realised lambda = n / log N (the
    rounding to integers makes it drift slightly from the requested grid), and the (R, n_grid)
    array of ARI sample paths.
    """
    rng = np.random.default_rng(seed)
    runs = []
    for N in N_list:
        n_grid = np.unique(np.maximum(np.round(lambdas * np.log(N)), 2).astype(np.int64))
        R = replicates_for(N)
        ari = np.empty((R, n_grid.size))
        t0 = time.time()
        for r in tqdm(range(R), desc=f"N = {N}", leave=False):
            mix = sample_mixture(K, N, int(n_grid[-1]), D_STATES, alpha, rng, kernels=kernels)
            ari[r] = evaluate_paths(mix["X"], mix["labels"], n_grid, K)["ari"]
        runs.append({"N": N, "n": n_grid, "lam": n_grid / np.log(N), "ari": ari})
        print(f"  N = {N:5d}: R = {R:3d}, n from {n_grid[0]:4d} to {n_grid[-1]:4d}, "
              f"{time.time() - t0:5.1f}s")
    return runs


t0 = time.time()
runs = sweep_n_and_N(N_LIST, LAMBDAS, KERNELS, ALPHA_FIX, K_FIX)
print(f"total: {time.time() - t0:.0f}s")

In [ ]:
def crossing(x, y, level=0.5):
    """Abscissa at which y first reaches `level`, by linear interpolation."""
    y = np.asarray(y, dtype=float)
    idx = int(np.argmax(y >= level))
    if y[idx] < level or idx == 0:
        return np.nan
    x0, x1, y0, y1 = x[idx - 1], x[idx], y[idx - 1], y[idx]
    return float(x0 + (level - y0) * (x1 - x0) / (y1 - y0))


def _cv(v):
    """Coefficient of variation, ignoring the entries that never crossed."""
    v = np.asarray(v, dtype=float)
    v = v[np.isfinite(v)]
    return float(np.std(v) / np.mean(v)) if v.size > 1 else np.nan


print(f"{'N':>6}  {'log N':>6}  {'n at ARI = 0.5':>14}  {'lambda at ARI = 0.5':>19}  "
      f"{'ARI at largest n':>16}")
n50, lam50 = [], []
for run in runs:
    mean_ari = run["ari"].mean(0)
    a = crossing(run["n"], mean_ari)
    b = crossing(run["lam"], mean_ari)
    n50.append(a)
    lam50.append(b)
    print(f"{run['N']:>6}  {np.log(run['N']):>6.2f}  {a:>14.1f}  {b:>19.2f}  {mean_ari[-1]:>16.3f}")

print(f"\ncoefficient of variation across N:  n = {_cv(n50):.3f}   "
      f"n / log N = {_cv(lam50):.3f}")
print("The rescaling is the right one to the extent that the second is the smaller.")

In [ ]:
def plot_ari_collapse(runs, show_paths=False, filename=None):
    """The same ARI curves against n and against n / log N.

    Left panel: the raw horizon, where the curves are ordered by N -- a larger sample needs
    longer sequences, because the union bound runs over N(N-1)/2 pairs. Right panel: the
    rescaled horizon, the only combination of n and N the exact-recovery bound involves.
    """
    with plt.rc_context(PAPER_STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6), sharey=True)
        cols = plt.get_cmap(SEQUENTIAL_CMAP)(np.linspace(0.42, 0.98, len(runs)))
        for run, col in zip(runs, cols):
            mean_ari = run["ari"].mean(0)
            lo = np.quantile(run["ari"], 0.1, axis=0)
            hi = np.quantile(run["ari"], 0.9, axis=0)
            for ax, x in ((axes[0], run["n"]), (axes[1], run["lam"])):
                ax.fill_between(x, lo, hi, color=col, alpha=0.18, lw=0)
                if show_paths:
                    ax.plot(x, run["ari"][:12].T, color=col, lw=0.5, alpha=0.25)
                ax.plot(x, mean_ari, color=col, lw=1.8, label=rf"$N = {run['N']}$")
        axes[0].set_xlabel(r"$n$")
        axes[1].set_xlabel(r"$n\,/\,\log N$")
        axes[0].set_ylabel("mean ARI, single linkage")
        axes[0].set_title("raw horizon", fontsize=9)
        axes[1].set_title(r"rescaled horizon $\lambda = n/\log N$", fontsize=9)
        for ax in axes:
            ax.set_xscale("log")
            ax.set_ylim(-0.04, 1.04)
            ax.axhline(0.5, color="0.6", lw=0.7, ls=(0, (1, 3)), zorder=0)
        axes[1].legend(loc="upper left", ncol=2, columnspacing=1.2)
        fig.suptitle(rf"fixed mixture: $K = {K_FIX}$ components, $\alpha = {ALPHA_FIX}$; "
                     rf"band = 10th to 90th percentile over replicates", fontsize=9)
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(f"Figures/Recovery/{filename}.{ext}", bbox_inches="tight")
            print("figure written to", f"Figures/Recovery/{filename}.pdf")
        plt.show()
    return fig


_ = plot_ari_collapse(runs, filename="ari_collapse_single_linkage")

## The $(\alpha, K)$ grid

The same grid as `assumptions.ipynb`, so the two sets of heatmaps can be read cell against cell:
$\alpha \in \{0.1, \dots, 10\}$, $K \in \{2, \dots, 10\}$, new chains at every repetition. Two
horizons are evaluated on nested prefixes, which is nearly free since the cost is dominated by the
largest one.

The horizon is smaller here than the $n = 1500$ of `assumptions.ipynb`: a full $N \times N$ matrix
costs $\binom{N}{2}$ dissimilarities instead of $K + K(K-1)/2$, so the budget goes into $N$ rather
than into $n$. The bottom rows of the heatmaps ($\alpha \le 0.2$, where the relaxation time of the
kernels is largest) are therefore finite-$n$ statements even more than they were there.

In [ ]:
def sweep_ari(alphas, Ks, N, grid, R, seed=SEED + 2):
    """Mean ARI, exact-recovery rate and separation margin on the (alpha, K) grid.

    Returns a dict of (len(alphas), len(Ks), len(grid)) arrays. `ari` and `exact` describe the
    clustering, `margin` and `p_sep` the assumption, all four measured on the same data.
    """
    grid = np.atleast_1d(np.asarray(grid, dtype=np.int64))
    rng = np.random.default_rng(seed)
    shape = (len(alphas), len(Ks), grid.size)
    res = {key: np.empty(shape) for key in ("ari", "exact", "margin", "p_sep", "in", "out")}
    for i, alpha in enumerate(tqdm(alphas, desc="alpha")):
        for j, K in enumerate(Ks):
            acc = {key: np.empty((R, grid.size)) for key in ("ari", "in", "out", "out_max")}
            for r in range(R):
                mix = sample_mixture(K, N, int(grid[-1]), D_STATES, float(alpha), rng)
                out = evaluate_paths(mix["X"], mix["labels"], grid, K)
                for key in acc:
                    acc[key][r] = out[key]
            m = acc["out"] - acc["in"]
            res["ari"][i, j] = acc["ari"].mean(axis=0)
            res["exact"][i, j] = (acc["ari"] > 1 - EXACT_TOL).mean(axis=0)
            res["margin"][i, j] = np.median(m, axis=0)
            res["p_sep"][i, j] = np.mean(m > 0, axis=0)
            res["in"][i, j] = np.median(acc["in"], axis=0)
            res["out"][i, j] = np.median(acc["out"], axis=0)
    return res, grid

In [ ]:
ALPHAS  = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 1.0, 5.0, 10.0])
KS      = [2, 3, 4, 5, 6, 7, 8, 9, 10]
N_SWEEP = 60                      # sequences per repetition
GRID_N  = np.array([150, 600])    # horizons, nested prefixes
R_SWEEP = 15                      # repetitions per cell

t0 = time.time()
sweep, GRID_N = sweep_ari(ALPHAS, KS, N=N_SWEEP, grid=GRID_N, R=R_SWEEP)
print(f"sweep: {time.time() - t0:.0f}s")

LAST = GRID_N.size - 1            # index of the largest horizon
for key, lab in (("ari", "mean ARI"), ("exact", "P(ARI = 1)"), ("p_sep", "P(margin > 0)")):
    Z = sweep[key][:, :, LAST]
    print(f"{lab:16s} in [{Z.min():.2f}, {Z.max():.2f}]")
print(f"{'median margin':16s} in [{sweep['margin'][:, :, LAST].min():+.2f}, "
      f"{sweep['margin'][:, :, LAST].max():+.2f}]")

In [ ]:
def _frame(ax, alphas, Ks, title):
    ax.set_xticks(range(len(Ks)), [str(K) for K in Ks])
    ax.set_yticks(range(len(alphas)), [f"{a:g}" for a in alphas])
    ax.set_xlabel(r"$K$")
    ax.set_ylabel(r"$\alpha$")
    ax.set_title(title, fontsize=9, pad=6)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color("0.7")


def _heat(fig, ax, Z, alphas, Ks, title, cbar_label, below=None, cmap=SEQUENTIAL_CMAP,
          vmin=0.0, vmax=1.0, fmt="{:.2f}", fmt_below="{:+.2f}"):
    """One heatmap of the (alpha, K) grid, optionally annotated with a second quantity below
    the first, in smaller type."""
    im = ax.imshow(Z, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    _frame(ax, alphas, Ks, title)
    span = max(abs(vmin), abs(vmax))
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            light = (Z[i, j] - vmin) / (vmax - vmin) > 0.55 if cmap == SEQUENTIAL_CMAP \
                else abs(Z[i, j]) > 0.62 * span
            col = "white" if light else "0.25"
            dy = 0.13 if below is not None else 0.0
            ax.text(j, i + dy, fmt.format(Z[i, j]), ha="center", va="center",
                    fontsize=5.5, color=col)
            if below is not None:
                ax.text(j, i - 0.17, fmt_below.format(below[i, j]), ha="center", va="center",
                        fontsize=4.6, color=col, alpha=0.85)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label(cbar_label, fontsize=8)
    cb.outline.set_visible(False)
    return im


def plot_ari_grid(sweep, alphas, Ks, n, g=LAST, filename=None):
    """Mean ARI and exact-recovery rate on the grid, each annotated with the median
    separation margin measured on the same data."""
    with plt.rc_context(PAPER_STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.6))
        _heat(fig, axes[0], sweep["ari"][:, :, g], alphas, Ks,
              rf"mean ARI, single linkage ($n = {n}$)", "mean ARI",
              below=sweep["margin"][:, :, g])
        _heat(fig, axes[1], sweep["exact"][:, :, g], alphas, Ks,
              rf"$\widehat{{\mathbb{{P}}}}(\mathrm{{ARI}} = 1)$, exact recovery ($n = {n}$)",
              "probability of exact recovery", below=sweep["margin"][:, :, g])
        fig.tight_layout()
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(f"Figures/Recovery/{filename}.{ext}", bbox_inches="tight")
            print("figure written to", f"Figures/Recovery/{filename}.pdf")
        plt.show()
    return fig


_ = plot_ari_grid(sweep, ALPHAS, KS, int(GRID_N[LAST]), g=LAST,
                  filename="ari_grid_single_linkage")

The small figure below repeats the mean ARI at the two horizons: whatever the cell, going from
$n = 150$ to $n = 600$ moves the ARI up, which is the consistency statement read across the whole
grid rather than at one model.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, GRID_N.size, figsize=(4.3 * GRID_N.size, 3.6))
    for g, ax in enumerate(np.atleast_1d(axes)):
        _heat(fig, ax, sweep["ari"][:, :, g], ALPHAS, KS,
              rf"mean ARI at $n = {int(GRID_N[g])}$", "mean ARI")
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(f"Figures/Recovery/ari_grid_horizons.{ext}", bbox_inches="tight")
    plt.show()

## Is the separation condition necessary?

Each point below is one cell of the grid: on the horizontal axis the median separation margin
$\hat\eta = \hat\Delta_{\mathrm{out}} - \hat\Delta_{\mathrm{in}}$, on the vertical axis the mean
ARI, both measured on the same repetitions. The condition being *sufficient* means the top-right
quadrant is populated; the question is the **top-left** one — cells where the margin is negative,
so the theorem says nothing, and single linkage recovers the partition anyway.

In [ ]:
def plot_ari_vs_margin(sweep, alphas, Ks, n, g=LAST, filename=None):
    """Mean ARI against the median separation margin, one marker per cell of the grid.

    The same scatter twice, coloured by K on the left and by alpha on the right: the two
    panels say which of the two directions of the grid drives a cell out of the region the
    theorem covers.
    """
    shape = sweep["ari"][:, :, g].shape
    x = sweep["margin"][:, :, g].ravel()
    y = sweep["ari"][:, :, g].ravel()
    by_k = np.broadcast_to(np.asarray(Ks, dtype=float)[None, :], shape).ravel()
    by_a = np.broadcast_to(np.arange(len(alphas), dtype=float)[:, None], shape).ravel()
    xlo = min(x.min() - 0.03, -0.02)
    with plt.rc_context(PAPER_STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6), sharey=True)
        specs = ((by_k, r"$K$", np.asarray(Ks, dtype=float),
                  [str(K) for K in Ks], min(Ks) - 1.2, max(Ks)),
                 (by_a, r"$\alpha$", np.arange(len(alphas), dtype=float),
                  [f"{a:g}" for a in alphas], -1.2, len(alphas) - 1))
        for ax, (c, lab, ticks, ticklabels, vmin, vmax) in zip(axes, specs):
            ax.axvspan(xlo, 0.0, color="0.9", lw=0, zorder=0)
            ax.axvline(0.0, color="0.4", lw=0.9, zorder=1)
            sc = ax.scatter(x, y, c=c, cmap=SEQUENTIAL_CMAP, vmin=vmin, vmax=vmax,
                            s=26, lw=0.4, edgecolor="0.35", zorder=2)
            cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
            cb.set_label(lab, fontsize=9)
            cb.set_ticks(ticks, labels=ticklabels)
            cb.outline.set_visible(False)
            ax.set_xlim(xlo, x.max() + 0.03)
            ax.set_xlabel(r"median margin "
                          r"$\hat\eta = \hat\Delta_{\mathrm{out}} - \hat\Delta_{\mathrm{in}}$")
            ax.text(0.5 * xlo, 0.5, "condition violated", rotation=90, ha="center", va="center",
                    fontsize=7, color="0.45", transform=ax.get_xaxis_transform())
        axes[0].set_ylabel("mean ARI")
        axes[0].set_ylim(-0.04, 1.04)
        fig.suptitle(rf"one marker per $(\alpha, K)$ cell, $n = {n}$, $N = {N_SWEEP}$",
                     fontsize=9)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(f"Figures/Recovery/{filename}.{ext}", bbox_inches="tight")
            print("figure written to", f"Figures/Recovery/{filename}.pdf")
        plt.show()
    return fig


_ = plot_ari_vs_margin(sweep, ALPHAS, KS, int(GRID_N[LAST]), g=LAST,
                       filename="ari_vs_separation")

In [ ]:
# how far the recovery extends past the region the theorem covers
margin = sweep["margin"][:, :, LAST]
ari = sweep["ari"][:, :, LAST]
p_sep = sweep["p_sep"][:, :, LAST]
violated = margin <= 0

mean_violated = float(ari[violated].mean()) if violated.any() else float("nan")

print(f"cells with median margin <= 0                 : {violated.sum()} / {margin.size}")
print(f"  of which mean ARI >= 0.9                    : {int((ari[violated] >= 0.9).sum())}")
print(f"  of which mean ARI >= 0.5                    : {int((ari[violated] >= 0.5).sum())}")
print(f"  mean ARI over these cells                   : {mean_violated:.2f}")
print(f"cells with P(margin > 0) <= 0.5 but ARI >= 0.9 : "
      f"{int(((p_sep <= 0.5) & (ari >= 0.9)).sum())}")
print()
print("mean ARI per K, averaged over alpha:")
for j, K in enumerate(KS):
    print(f"  K = {K:2d}: mean ARI = {ari[:, j].mean():.2f}   "
          f"P(ARI = 1) = {sweep['exact'][:, j, LAST].mean():.2f}   "
          f"median margin = {margin[:, j].mean():+.2f}")